# 04 — Sensibilidad hipotética de costos en validation

Este notebook calcula escenarios **hipotéticos, no aprobados** de BMR binario frente al umbral fijo 0,5. Lee únicamente los scores de `validation_policy` producidos por notebook 03. No reentrena ni carga test. El cálculo y la validación compartidos están en `src/fraud_cost/scenarios.py`.

## Supuestos explícitos

- Unidad: unidad registrada en `TransactionAmt`; no se etiqueta USD porque falta confirmar la fuente.
- `λ ∈ {0,25; 0,50; 1,00}`: fracción hipotética de pérdida si se aprueba fraude.
- `B ∈ {0,25; 0,50; 1,00; 2,00}` veces la mediana de monto de `validation_policy` (sin filtrar por etiqueta) como costo hipotético de bloquear una legítima. Esta referencia es descriptiva y retrospectiva.
- `C_block_fraud = 0`: supuesto idealizado de prevención completa, no hecho observado.
- Revisión y árbol sensibles al costo están fuera de esta sensibilidad.

La tabla mide costo proxy en las mismas observaciones de `validation_policy` usadas para medir el resultado. No representa ahorro real ni estimación independiente fuera de muestra. Con `C=0` y sin revisión, los 12 puntos de la grilla generan seis políticas distintas porque las decisiones dependen de `λ/B`.

La elección histórica del gap consultó conteos agregados de fraude de los candidatos a test; el test actual no es un holdout estrictamente intocado. Este notebook no carga ni puntúa ese bloque.

## 1. Cargar scores y validar alcance

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "interim").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Abra Jupyter desde el repositorio o una subcarpeta")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from fraud_cost.scenarios import evaluate_hypothetical_scenarios, validate_validation_scores
SCORES_PATH = PROJECT_ROOT / "data" / "interim" / "validation_scores.parquet"
TABLE_PATH = PROJECT_ROOT / "results" / "tables" / "escenarios_hipoteticos_validation.csv"
FIGURE_PATH = PROJECT_ROOT / "results" / "figures" / "ahorro_hipotetico_validation.png"
if not SCORES_PATH.exists():
    raise FileNotFoundError(f"No existe {SCORES_PATH}; ejecute notebook 03 hasta métricas predictivas")

scores = pd.read_parquet(SCORES_PATH)
amount, y, p, action_fixed, reference_amount = validate_validation_scores(scores)
print(f"validation_policy: {len(scores):,} filas; {int(y.sum()):,} fraudes; mediana descriptiva {reference_amount:.4f} unidades")
print("La entrada es la ventana posterior de validation; no se cargó test ni se reentrenó el modelo.")

## 2. Evaluar la cuadrícula de escenarios

In [ ]:
LAMBDA_GRID = (0.25, 0.50, 1.00)
B_RATIO_GRID = (0.25, 0.50, 1.00, 2.00)
FRAUD_BLOCK_RESIDUAL_COST = 0.0
SCENARIO_STATUS = "HIPOTETICO_NO_APROBADO"

scenario_results = evaluate_hypothetical_scenarios(
    scores,
    lambda_grid=LAMBDA_GRID,
    b_ratio_grid=B_RATIO_GRID,
    fraud_block_residual_cost=FRAUD_BLOCK_RESIDUAL_COST,
)
bmr_results = scenario_results[scenario_results["strategy"].eq("bmr_binary")].copy()
print(f"Escenarios BMR: {len(bmr_results)}; todos marcados como {SCENARIO_STATUS}.")
display(bmr_results[[
    "loss_fraction_lambda",
    "legitimate_block_cost_ratio_of_validation_median_amount",
    "legitimate_block_cost_B_proxy",
    "cost_total_proxy", "fixed_baseline_cost_proxy",
    "savings_vs_fixed_proxy", "savings_fraction_vs_fixed_proxy",
    "fraud_recall", "legitimate_block_rate",
]])

## 3. Guardar tabla y gráfica

In [ ]:
TABLE_PATH.parent.mkdir(parents=True, exist_ok=True)
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
scenario_results.to_csv(TABLE_PATH, index=False)

savings = (
    bmr_results.pivot(
        index="loss_fraction_lambda",
        columns="legitimate_block_cost_ratio_of_validation_median_amount",
        values="savings_fraction_vs_fixed_proxy",
    ).sort_index()
)
fig, ax = plt.subplots(figsize=(8, 4.8))
image = ax.imshow(savings.to_numpy(), cmap="RdYlGn", aspect="auto", vmin=-1, vmax=1)
ax.set_xticks(range(len(savings.columns)), [f"{x:g}×" for x in savings.columns])
ax.set_yticks(range(len(savings.index)), [f"λ={x:g}" for x in savings.index])
ax.set_xlabel("B hipotético / mediana en validation_policy")
ax.set_ylabel("Fracción de pérdida hipotética λ")
ax.set_title("Ahorro proxy hipotético de BMR frente a umbral 0,5\nvalidation_policy; C bloqueo fraude = 0")
for i in range(savings.shape[0]):
    for j in range(savings.shape[1]):
        ax.text(j, i, f"{savings.iloc[i, j]:.1%}", ha="center", va="center", color="black")
fig.colorbar(image, ax=ax, label="Ahorro relativo proxy vs. fijo")
fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()
print(f"Tabla guardada: {TABLE_PATH}")
print(f"Figura guardada: {FIGURE_PATH}")

## 4. Interpretación y límites

El ahorro relativo de cada escenario es `(costo_fijo - costo_BMR) / costo_fijo`, solo si el costo fijo es positivo. El recall y la tasa de legítimas bloqueadas muestran el intercambio operativo de cada regla.

No interprete estos valores como ahorro financiero real: dependen de supuestos hipotéticos y se miden sobre validation_policy, la misma ventana usada para inspeccionar el modelo y definir la mediana de B. La unidad de monto aún no está confirmada. Este notebook no modifica `COST_CONFIG`, no aprueba parámetros, no evalúa revisión ni el árbol y no abre el gate de test.

Los outputs previos se limpiaron al extraer la implementación compartida. Ejecute otra vez el notebook después de 03 para regenerar tabla y figura con el código actual.

Para un resultado final se requieren costos sustentados/aprobados, una política congelada y una sola evaluación en test.